# Download Dryad-linked article PDFs (Improved)

This notebook retrieves article PDFs for Dryad datasets with fallback strategies:
- **Strategy 1**: OpenAlex PDF URL
- **Strategy 2**: Unpaywall API fallback
- **Strategy 3**: Proxy retry (for university VPN access)

**Improvements over original:**
- Eliminates redundant OpenAlex API calls
- Uses `download_pdf_with_fallback()` for robust PDF retrieval
- Caches OpenAlex work objects
- Cleaner, more maintainable code

In [2]:
# Section 1: Imports and configuration
from pathlib import Path
import pandas as pd
import logging

from llm_metadata.openalex import get_work_by_doi, extract_pdf_url
from llm_metadata.pdf_download import download_pdf_with_fallback
from llm_metadata.ezproxy import extract_cookies_from_browser

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger("download_dryad_pdfs")

# Output directory
OUTPUT_DIR = Path("data/pdfs/fuster")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Try to extract cookies for EZProxy
COOKIES = extract_cookies_from_browser(browser="firefox")

# Manifest path
MANIFEST_PATH = OUTPUT_DIR / "manifest.csv"

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Manifest path: {MANIFEST_PATH}")

Extracting cookies from Firefox...
  ✓ Found 2 EZproxy cookie(s)
  ✓ Found 6 CAS cookie(s)
✓ Total: 8 cookie(s)
Output directory: C:\Users\beav3503\dev\llm_metadata\data\pdfs\fuster
Manifest path: data\pdfs\fuster\manifest.csv


## Section 2: Load dataset-article mapping and select sample

Load Dryad datasets with article DOIs and select a sample for testing.

In [3]:
# Load mapping CSV
mapping_path = Path("data/dataset_article_mapping.csv")
assert mapping_path.exists(), f"Mapping file not found: {mapping_path}"

df = pd.read_csv(mapping_path, dtype=str)
df.columns = [c.strip() for c in df.columns]

# Filter: Dryad rows with article DOI
df['article_doi'] = df['article_doi'].fillna('').str.strip()
df['source'] = df['source'].fillna('').str.strip().str.lower()
dryad_df = df[(df['source'] == 'dryad') & (df['article_doi'] != '')].copy()
dryad_df.reset_index(drop=True, inplace=True)

print(f"Found {len(dryad_df)} Dryad datasets with article DOIs")

# Select sample (adjust SAMPLE_SIZE as needed)
SAMPLE_SIZE = 5
sample_df = dryad_df.sample(n=min(SAMPLE_SIZE, len(dryad_df)), random_state=38)
sample_df = sample_df.reset_index(drop=True)

print(f"\nSelected sample ({len(sample_df)} works):")
sample_df[['dataset_doi', 'article_doi', 'title']]

Found 37 Dryad datasets with article DOIs

Selected sample (5 works):


,dataset_doi,article_doi,title
0,10.5061/dryad.4k275,10.1186/s40462-016-0079-4,"Data from: Caribou, water, and ice – fine-scal..."
1,10.5061/dryad.qt30h,10.1111/1365-2664.12517,Data from: The phylogenetics of succession can...
2,10.5061/dryad.1771t,10.1371/journal.pone.0128238,Data from: Resampling method for applying dens...
3,10.5061/dryad.d5j34,10.1098/rspb.2014.1779,Data from: Non-climatic constraints on upper e...
4,10.5061/dryad.s26qt314,10.1111/j.1420-9101.2011.02418.x,Data from: Regional divergence and mosaic spat...


## Section 3: Fetch OpenAlex works (single pass)

Fetch OpenAlex metadata **once** for each DOI and cache the results.

In [4]:
import time

# Fetch OpenAlex works (single pass)
works_cache = {}

for idx, row in sample_df.iterrows():
    doi = row['article_doi']
    
    logger.info(f"Fetching OpenAlex work for DOI: {doi}")
    
    try:
        work = get_work_by_doi(doi)
        works_cache[doi] = work
        
        if work:
            oa_status = work.get('open_access', {}).get('oa_status', 'closed')
            is_oa = work.get('open_access', {}).get('is_oa', False)
            logger.info(f"  ✓ Found work - OA Status: {oa_status}, Is OA: {is_oa}")
        else:
            logger.warning(f"  ✗ Work not found")
    
    except Exception as e:
        logger.error(f"  ✗ Error: {e}")
        works_cache[doi] = None
    
    # Be polite - sleep between requests
    time.sleep(1)

print(f"\n✓ Fetched {len(works_cache)} OpenAlex works")
print(f"  Found: {sum(1 for w in works_cache.values() if w is not None)}")
print(f"  Missing: {sum(1 for w in works_cache.values() if w is None)}")

INFO:download_dryad_pdfs:Fetching OpenAlex work for DOI: 10.1186/s40462-016-0079-4
INFO:download_dryad_pdfs:  ✓ Found work - OA Status: gold, Is OA: True
INFO:download_dryad_pdfs:Fetching OpenAlex work for DOI: 10.1111/1365-2664.12517
INFO:download_dryad_pdfs:  ✓ Found work - OA Status: bronze, Is OA: True
INFO:download_dryad_pdfs:Fetching OpenAlex work for DOI: 10.1371/journal.pone.0128238
INFO:download_dryad_pdfs:  ✓ Found work - OA Status: gold, Is OA: True
INFO:download_dryad_pdfs:Fetching OpenAlex work for DOI: 10.1098/rspb.2014.1779
INFO:download_dryad_pdfs:  ✓ Found work - OA Status: bronze, Is OA: True
INFO:download_dryad_pdfs:Fetching OpenAlex work for DOI: 10.1111/j.1420-9101.2011.02418.x
INFO:download_dryad_pdfs:  ✓ Found work - OA Status: bronze, Is OA: True



✓ Fetched 5 OpenAlex works
  Found: 5
  Missing: 0


## Section 4: Download PDFs with fallback strategies

Use `download_pdf_with_fallback()` to try multiple strategies:
1. OpenAlex PDF URL
2. Unpaywall API fallback
3. Proxy retry (if configured in `.env`)

In [5]:
results = []

for idx, row in sample_df.iterrows():
    doi = row['article_doi']
    dataset_doi = row['dataset_doi']
    title = row.get('title', '')
    
    work = works_cache.get(doi)
    
    record = {
        'article_doi': doi,
        'dataset_doi': dataset_doi,
        'title': title,
        'openalex_id': work.get('id') if work else None,
        'oa_status': work.get('open_access', {}).get('oa_status') if work else None,
        'openalex_pdf_url': None,
        'downloaded_pdf_path': None,
        'status': 'pending',
        'error': None
    }
    
    # Skip if no OpenAlex work
    if not work:
        record['status'] = 'no_openalex_work'
        record['error'] = 'OpenAlex work not found'
        results.append(record)
        continue
    
    # Extract OpenAlex PDF URL (may be None)
    openalex_pdf_url = extract_pdf_url(work)
    record['openalex_pdf_url'] = openalex_pdf_url
    
    # Attempt download with fallback
    logger.info(f"\nDownloading PDF for {doi}...")
    
    try:
        pdf_path = download_pdf_with_fallback(
            doi=doi,
            openalex_pdf_url=openalex_pdf_url,
            output_dir=OUTPUT_DIR,
            use_unpaywall=True,
            ezproxy_cookies=COOKIES
        )
        
        if pdf_path:
            record['status'] = 'downloaded'
            record['downloaded_pdf_path'] = str(pdf_path)
            logger.info(f"  ✓ Downloaded successfully: {pdf_path.name}")
        else:
            record['status'] = 'failed'
            record['error'] = 'All download strategies failed'
            logger.warning(f"  ✗ All download strategies failed")
    
    except Exception as e:
        record['status'] = 'error'
        record['error'] = str(e)
        logger.error(f"  ✗ Exception: {e}")
    
    results.append(record)
    
    # Be polite between downloads
    time.sleep(0.5)

# Build results DataFrame
results_df = pd.DataFrame(results)
results_df

INFO:download_dryad_pdfs:
INFO:llm_metadata.pdf_download:PDF already exists: data\pdfs\fuster\10.1186_s40462-016-0079-4.pdf
INFO:download_dryad_pdfs:  ✓ Downloaded successfully: 10.1186_s40462-016-0079-4.pdf
INFO:download_dryad_pdfs:
INFO:llm_metadata.pdf_download:Strategy 1: Trying OpenAlex PDF URL for 10.1111/1365-2664.12517
INFO:llm_metadata.pdf_download:Downloading from https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/1365-2664.12517 (attempt 1/3)
INFO:llm_metadata.pdf_download:Downloading from https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/1365-2664.12517 (attempt 2/3)
INFO:llm_metadata.pdf_download:Downloading from https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/1365-2664.12517 (attempt 3/3)
INFO:llm_metadata.pdf_download:Strategy 2: Trying Unpaywall for 10.1111/1365-2664.12517
INFO:llm_metadata.pdf_download:Strategy 3: Trying EZproxy for 10.1111/1365-2664.12517
INFO:llm_metadata.pdf_download:Trying EZproxy proxied_publisher for 10.1111/1365-2664.12517
INFO:llm_

,article_doi,dataset_doi,title,openalex_id,oa_status,openalex_pdf_url,downloaded_pdf_path,status,error
0,10.1186/s40462-016-0079-4,10.5061/dryad.4k275,"Data from: Caribou, water, and ice – fine-scal...",https://openalex.org/W2339707562,gold,https://movementecologyjournal.biomedcentral.c...,data\pdfs\fuster\10.1186_s40462-016-0079-4.pdf,downloaded,None
1,10.1111/1365-2664.12517,10.5061/dryad.qt30h,Data from: The phylogenetics of succession can...,https://openalex.org/W2156526425,bronze,https://onlinelibrary.wiley.com/doi/pdfdirect/...,data\pdfs\fuster\10.1111_1365-2664.12517.pdf,downloaded,None
2,10.1371/journal.pone.0128238,10.5061/dryad.1771t,Data from: Resampling method for applying dens...,https://openalex.org/W611491694,gold,https://journals.plos.org/plosone/article/file...,data\pdfs\fuster\10.1371_journal.pone.0128238.pdf,downloaded,None
3,10.1098/rspb.2014.1779,10.5061/dryad.d5j34,Data from: Non-climatic constraints on upper e...,https://openalex.org/W2138744437,bronze,https://royalsocietypublishing.org/doi/pdf/10....,data\pdfs\fuster\10.1098_rspb.2014.1779.pdf,downloaded,None
4,10.1111/j.1420-9101.2011.02418.x,10.5061/dryad.s26qt314,Data from: Regional divergence and mosaic spat...,https://openalex.org/W1572281480,bronze,https://onlinelibrary.wiley.com/doi/pdfdirect/...,data\pdfs\fuster\10.1111_j.1420-9101.2011.0241...,downloaded,None


## Section 5: Summary and save manifest

In [6]:
# Save manifest
results_df.to_csv(MANIFEST_PATH, index=False)

# Summary statistics
summary = results_df['status'].value_counts()

print("\n" + "="*60)
print("DOWNLOAD SUMMARY")
print("="*60)

for status, count in summary.items():
    print(f"{status:20s}: {count}")

print(f"\nManifest saved to: {MANIFEST_PATH}")

# Show downloaded files
downloaded = results_df[results_df['status'] == 'downloaded']
if len(downloaded) > 0:
    print(f"\n✓ Successfully downloaded {len(downloaded)} PDFs:")
    for _, row in downloaded.iterrows():
        print(f"  - {row['article_doi']} → {Path(row['downloaded_pdf_path']).name}")
else:
    print("\n✗ No PDFs were downloaded.")
    print("\nTroubleshooting tips:")
    print("1. Check if OPENALEX_EMAIL is set in .env (for Unpaywall)")
    print("2. Configure HTTP_PROXY and HTTPS_PROXY if you have university VPN access")
    print("3. Some publishers block direct PDF access - this is expected")

# Show failures with details
failed = results_df[results_df['status'].isin(['failed', 'error'])]
if len(failed) > 0:
    print(f"\n✗ Failed downloads ({len(failed)}):")
    for _, row in failed.iterrows():
        print(f"  - {row['article_doi']}")
        print(f"    Error: {row['error']}")
        print(f"    OA Status: {row['oa_status']}")


DOWNLOAD SUMMARY
downloaded          : 5

Manifest saved to: data\pdfs\fuster\manifest.csv

✓ Successfully downloaded 5 PDFs:
  - 10.1186/s40462-016-0079-4 → 10.1186_s40462-016-0079-4.pdf
  - 10.1111/1365-2664.12517 → 10.1111_1365-2664.12517.pdf
  - 10.1371/journal.pone.0128238 → 10.1371_journal.pone.0128238.pdf
  - 10.1098/rspb.2014.1779 → 10.1098_rspb.2014.1779.pdf
  - 10.1111/j.1420-9101.2011.02418.x → 10.1111_j.1420-9101.2011.02418.x.pdf
